# RSNA Pneumonia Detection (Stage 2) — Binary ResNet50 Classifier for the ED²A `advanced_pneumonia` Service

## Objective

Train an end-to-end binary chest X-ray classifier (**pneumonia vs. no pneumonia**) on the
Kaggle *RSNA Pneumonia Detection Challenge — Stage 2* dataset using a pretrained
**ResNet50**, and export a self-contained deployment bundle for the future ED²A
`advanced_pneumonia` inference service.

The pipeline covers: annotation loading and validation, exploratory data analysis,
a reproducible patient-level stratified 70/30 split, robust DICOM decoding, medically
plausible augmentation, class-imbalance handling, full fine-tuning with early stopping,
validation-only evaluation, and artifact export (PyTorch checkpoint, ONNX graph,
configuration, metrics, manifest, requirements and a ZIP bundle).

## Important limitation of the medical label

In `stage_2_train_labels.csv`, `Target = 1` marks an image in which a radiologist
annotated **a pulmonary opacity compatible with pneumonia** — it is a bounding-box
annotation of a radiological finding, **not a definitive clinical diagnosis of pneumonia**.
A confirmed diagnosis requires clinical context (symptoms, vitals, laboratory results,
history) that is entirely absent from this dataset. Consequently:

- The positive class must be read as *"radiographic opacity suspicious for pneumonia"*.
- Model outputs are **decision support for triage experiments only**, never a diagnosis.
- Any downstream ED²A integration must present the result with an explicit disclaimer,
  consistent with the disclaimer already used by the DDXPlus differential-diagnosis model.

## Scope of this notebook

- Only the **labelled Stage 2 training set** is used, split into train/validation.
  The official Stage 2 *test* images are **unlabelled**, so no external test-set
  evaluation is produced here — every reported metric comes from the held-out
  validation split.
- Multiple CSV rows per `patientId` (one per bounding box) are aggregated into
  **exactly one binary record per patient**. Bounding-box coordinates are never used
  as model inputs.
- Nothing in this notebook has been executed: every cell must be run by the user after
  the two dataset placeholders in the configuration cell are replaced.

---
## 2. Dependencies and reproducibility

The first cell is the **only** dependency-installation cell and it is optional: it stays
inert until `INSTALL_DEPENDENCIES` is set to `True`. No local wheel files are used.

In [ ]:
"""Optional, isolated dependency installation.

Set INSTALL_DEPENDENCIES = True the first time this notebook runs in a fresh
environment. While the flag is False nothing is installed, so an already working
environment is never mutated by accident.
"""

import subprocess
import sys

INSTALL_DEPENDENCIES = False

RUNTIME_PACKAGES = [
    "torch",
    "torchvision",
    "pydicom",
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "pillow",
    "onnx",
    "onnxruntime",
]

if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", *RUNTIME_PACKAGES])
    print("Dependencies installed. Restart the kernel if a core library was upgraded.")
else:
    print("Dependency installation skipped (INSTALL_DEPENDENCIES = False).")

In [ ]:
"""Imports, version reporting, seeding and version-safe AMP helpers."""

import copy
import hashlib
import importlib
import io
import json
import math
import os
import platform
import random
import time
import zipfile
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50

try:  # pydicom >= 3.0
    from pydicom.pixels import apply_voi_lut
except ImportError:  # pydicom < 3.0
    from pydicom.pixel_data_handlers.util import apply_voi_lut


def _module_version(module_name: str) -> str:
    """Return a module version string without failing on optional dependencies."""
    try:
        module = importlib.import_module(module_name)
    except Exception:
        return "not installed"
    return str(getattr(module, "__version__", "unknown"))


def library_versions() -> dict[str, str]:
    """Collect the library versions that describe this training environment."""
    return {
        "python": platform.python_version(),
        "torch": _module_version("torch"),
        "torchvision": _module_version("torchvision"),
        "numpy": _module_version("numpy"),
        "pandas": _module_version("pandas"),
        "scikit-learn": _module_version("sklearn"),
        "pydicom": _module_version("pydicom"),
        "pillow": _module_version("PIL"),
        "onnx": _module_version("onnx"),
        "onnxruntime": _module_version("onnxruntime"),
    }


def set_seed(seed: int) -> None:
    """Seed every random source used by this notebook."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int) -> None:
    """Make DataLoader worker randomness reproducible across runs."""
    worker_seed = (torch.initial_seed() + worker_id) % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_grad_scaler(enabled: bool) -> Any:
    """Build a GradScaler that works with both the new and the legacy torch AMP API."""
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def amp_autocast(device_type: str, enabled: bool) -> Any:
    """Return an autocast context manager compatible with new and legacy torch."""
    try:
        return torch.amp.autocast(device_type=device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)


for _name, _version in library_versions().items():
    print(f"{_name:>14}: {_version}")
print(f"{'cuda available':>14}: {torch.cuda.is_available()}")

---
## 3. Configuration

Every constant used by the notebook lives in the single cell below.

> **Action required before execution.**
> Replace the two `PLACEHOLDER` dataset paths with the real locations on your machine
> or Kaggle environment:
> - `LABELS_CSV` → the full path of `stage_2_train_labels.csv`.
> - `TRAIN_DICOM_DIR` → the directory holding the Stage 2 **training** `.dcm` files.
>
> These are the **only** dataset input paths in the notebook. `ARTIFACT_DIR` is an output
> directory and is intentionally relative — it never points at the dataset and never
> points at the ED²A application `artifacts/` folder.

In [ ]:
"""Centralised configuration: dataset placeholders, hyperparameters and artifact names."""

# --- Dataset input paths -------------------------------------------------------------
# REPLACE BOTH PLACEHOLDERS BEFORE RUNNING THE NOTEBOOK.
# LABELS_CSV must point to stage_2_train_labels.csv.
# TRAIN_DICOM_DIR must point to the directory containing the Stage 2 training .dcm files.
LABELS_CSV = Path("PLACEHOLDER")
TRAIN_DICOM_DIR = Path("PLACEHOLDER")

# --- Artifact output path (not a dataset path) ---------------------------------------
ARTIFACT_DIR = Path("./ed2a_artifacts/advanced_pneumonia")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# --- Task definition -----------------------------------------------------------------
SERVICE_NAME = "advanced_pneumonia"
ARCHITECTURE = "resnet50"
SCHEMA_VERSION = 1
CLASS_NAMES: dict[int, str] = {0: "NO_PNEUMONIA", 1: "PNEUMONIA"}
POSITIVE_CLASS_INDEX = 1
DATASET_NAME = "RSNA Pneumonia Detection Challenge (Stage 2) - labelled training set"
LABEL_SEMANTICS = (
    "Target=1 marks a radiologist-annotated pulmonary opacity compatible with pneumonia; "
    "it is a radiological finding, not a confirmed clinical diagnosis."
)

# --- Reproducibility and hardware ----------------------------------------------------
SEED = 42
NUMBER_OF_GPU = 1
DEVICE = torch.device("cuda" if (torch.cuda.is_available() and NUMBER_OF_GPU >= 1) else "cpu")
USE_AMP = DEVICE.type == "cuda"  # mixed precision only makes sense on CUDA

# --- Data pipeline -------------------------------------------------------------------
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
PIN_MEMORY = DEVICE.type == "cuda"
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.30
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# --- Conservative, medically plausible augmentation ----------------------------------
ROTATION_DEGREES = 7.0            # small rotation only: chest radiographs have a fixed orientation
TRANSLATE_FRACTION = 0.05         # <= 5% shift keeps the whole thoracic field inside the frame
SCALE_RANGE = (0.95, 1.05)        # mild zoom
HORIZONTAL_FLIP_P = 0.5           # configurable; set to 0.0 to preserve left/right laterality
BRIGHTNESS_JITTER = 0.10
CONTRAST_JITTER = 0.10

# --- Model and optimisation ----------------------------------------------------------
MAX_EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.30
DECISION_THRESHOLD = 0.50
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MIN_DELTA = 1e-4
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3

# --- DICOM decoding ------------------------------------------------------------------
CLIP_PERCENTILES = (0.5, 99.5)

# --- ONNX export ---------------------------------------------------------------------
ONNX_OPSET = 17
ONNX_INPUT_NAME = "input"
ONNX_OUTPUT_NAME = "logit"
SUPPORTED_UPLOAD_FORMATS = ("JPEG", "PNG", "WEBP")

# --- Artifact file names -------------------------------------------------------------
TRAIN_SPLIT_FILENAME = "train_split.csv"
VALIDATION_SPLIT_FILENAME = "validation_split.csv"
BEST_CHECKPOINT_FILENAME = "advanced_pneumonia_best_epoch.pt"
CHECKPOINT_FILENAME = "advanced_pneumonia_checkpoint.pt"
ONNX_FILENAME = "advanced_pneumonia_model.onnx"
CONFIG_FILENAME = "advanced_pneumonia_config.json"
METRICS_FILENAME = "advanced_pneumonia_metrics.json"
MANIFEST_FILENAME = "advanced_pneumonia_manifest.json"
REQUIREMENTS_FILENAME = "advanced_pneumonia_requirements.txt"
BUNDLE_FILENAME = "ed2a_advanced_pneumonia_artifacts.zip"

BUNDLE_MEMBERS = [
    CHECKPOINT_FILENAME,
    ONNX_FILENAME,
    CONFIG_FILENAME,
    METRICS_FILENAME,
    REQUIREMENTS_FILENAME,
]
GENERATED_FILENAMES = [
    TRAIN_SPLIT_FILENAME,
    VALIDATION_SPLIT_FILENAME,
    BEST_CHECKPOINT_FILENAME,
    *BUNDLE_MEMBERS,
    MANIFEST_FILENAME,
    BUNDLE_FILENAME,
]

# --- ED2A artifact-name protection ---------------------------------------------------
# These names belong to the deployed DDXPlus differential-diagnosis model
# (API_App/artifacts/). This notebook must never create or overwrite them.
DDXPLUS_RESERVED_FILENAMES = frozenset(
    {
        "best_model.pkl",
        "preprocessor.pkl",
        "label_encoder.pkl",
        "model_metrics.json",
        "artifact_manifest.json",
    }
)
assert not (set(GENERATED_FILENAMES) & DDXPLUS_RESERVED_FILENAMES), (
    "A generated artifact name collides with a reserved DDXPlus artifact name."
)


def safe_artifact_path(filename: str) -> Path:
    """Resolve a file inside ARTIFACT_DIR, refusing every reserved DDXPlus artifact name."""
    if filename in DDXPLUS_RESERVED_FILENAMES:
        raise ValueError(
            f"'{filename}' is a DDXPlus differential-diagnosis artifact and must never "
            "be written by the pneumonia pipeline."
        )
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    return ARTIFACT_DIR / filename


def ensure_dataset_paths_configured() -> None:
    """Fail fast while the dataset placeholders have not been replaced."""
    for name, path in (("LABELS_CSV", LABELS_CSV), ("TRAIN_DICOM_DIR", TRAIN_DICOM_DIR)):
        if "PLACEHOLDER" in str(path):
            raise ValueError(
                f"{name} still contains 'PLACEHOLDER'. Edit the configuration cell and set "
                "it to the real dataset location before running the rest of the notebook."
            )
        if not path.exists():
            raise FileNotFoundError(f"{name} does not exist: {path}")


set_seed(SEED)
print(f"Device            : {DEVICE}")
print(f"Mixed precision   : {USE_AMP}")
print(f"Artifact directory: {ARTIFACT_DIR.resolve()}")
print(f"Labels CSV        : {LABELS_CSV}")
print(f"DICOM directory   : {TRAIN_DICOM_DIR}")

---
## 4. Annotation loading and validation

`stage_2_train_labels.csv` contains one row **per bounding box**, so a positive patient
appears several times. The loader below aggregates the CSV into exactly **one binary
record per patient**:

- group by `patientId`;
- aggregated label = `max(Target)` (any annotated opacity makes the patient positive);
- `0` → `NO_PNEUMONIA`, `1` → `PNEUMONIA`;
- bounding-box coordinates (`x`, `y`, `width`, `height`) are dropped and never used as inputs.

The loader asserts that `patientId` is unique after aggregation, that labels contain only
`0` and `1`, and that every constructed image path ends in `.dcm`. Missing DICOM files are
counted and reported instead of failing silently.

In [ ]:
"""Load stage_2_train_labels.csv and build one validated binary record per patient."""

REQUIRED_LABEL_COLUMNS = {"patientId", "Target"}


def load_patient_labels(
    labels_csv: Path,
    dicom_dir: Path,
    class_names: dict[int, str] = CLASS_NAMES,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Aggregate the RSNA bounding-box CSV into one binary record per patient.

    Args:
        labels_csv: Path to ``stage_2_train_labels.csv``.
        dicom_dir: Directory containing the Stage 2 training ``.dcm`` files.
        class_names: Mapping from binary label to human readable class name.

    Returns:
        A tuple ``(patients, report)`` where ``patients`` has one row per patient with
        columns ``patientId``, ``label``, ``class_name`` and ``dicom_path``, and
        ``report`` summarises the aggregation and the missing-file check.
    """
    raw = pd.read_csv(labels_csv)
    missing_columns = REQUIRED_LABEL_COLUMNS - set(raw.columns)
    if missing_columns:
        raise ValueError(f"{labels_csv} is missing required columns: {sorted(missing_columns)}")

    raw["patientId"] = raw["patientId"].astype(str)
    raw["Target"] = pd.to_numeric(raw["Target"], errors="raise").astype(int)

    rows_per_patient = raw["patientId"].value_counts()
    aggregated = (
        raw.groupby("patientId", as_index=False)["Target"]
        .max()
        .rename(columns={"Target": "label"})
    )
    aggregated["label"] = aggregated["label"].astype(int)

    assert aggregated["patientId"].is_unique, "patientId is not unique after aggregation."
    invalid_labels = sorted(set(aggregated["label"].unique()) - {0, 1})
    assert not invalid_labels, f"Aggregated labels must be 0 or 1, found: {invalid_labels}"

    aggregated["class_name"] = aggregated["label"].map(class_names)
    assert aggregated["class_name"].notna().all(), "Every label must map to a class name."

    aggregated["dicom_path"] = aggregated["patientId"].map(lambda pid: dicom_dir / f"{pid}.dcm")
    assert aggregated["dicom_path"].map(lambda p: p.suffix == ".dcm").all(), (
        "Every constructed image path must end in '.dcm'."
    )

    exists_mask = aggregated["dicom_path"].map(lambda p: p.exists())
    missing_frame = aggregated.loc[~exists_mask]
    patients = aggregated.loc[exists_mask].reset_index(drop=True)
    if patients.empty:
        raise FileNotFoundError(
            f"No DICOM file from the annotation CSV was found under {dicom_dir}."
        )

    report = {
        "csv_rows": int(len(raw)),
        "unique_patients_in_csv": int(raw["patientId"].nunique()),
        "patients_with_multiple_rows": int((rows_per_patient > 1).sum()),
        "max_rows_for_one_patient": int(rows_per_patient.max()),
        "patients_after_aggregation": int(len(aggregated)),
        "missing_dicom_files": int((~exists_mask).sum()),
        "missing_examples": [p.name for p in missing_frame["dicom_path"].head(10)],
        "usable_patients": int(len(patients)),
    }
    return patients, report


ensure_dataset_paths_configured()
patients_df, label_report = load_patient_labels(LABELS_CSV, TRAIN_DICOM_DIR)

print("Annotation aggregation report")
print("-" * 60)
for key, value in label_report.items():
    print(f"{key:>28}: {value}")

if label_report["missing_dicom_files"] > 0:
    print(
        f"\nWARNING: {label_report['missing_dicom_files']} DICOM file(s) referenced by the CSV "
        f"were not found under {TRAIN_DICOM_DIR} and were excluded."
    )
    print(f"First missing files: {label_report['missing_examples']}")
else:
    print("\nAll DICOM files referenced by the annotation CSV were found.")

patients_df.head()

---
## 5. Exploratory data analysis

Class counts, prevalence and the imbalance ratio are computed from the data at runtime —
nothing is hard-coded. The helper that renders a reproducible grid of decoded examples is
defined here, but it is **called at the end of section 7**, once the single canonical DICOM
decoder exists: this keeps exactly one decoding implementation in the notebook.

In [ ]:
"""Exploratory analysis helpers: class distribution, sampling and DICOM header inspection."""


def sample_per_class(frame: pd.DataFrame, n_per_class: int, seed: int = SEED) -> pd.DataFrame:
    """Draw a reproducible, balanced sample with ``n_per_class`` rows from each class."""
    smallest_class = int(frame["label"].value_counts().min())
    effective_n = max(1, min(n_per_class, smallest_class))
    return (
        frame.groupby("label", group_keys=False)
        .sample(n=effective_n, random_state=seed)
        .sort_values("label")
        .reset_index(drop=True)
    )


def class_distribution(frame: pd.DataFrame) -> pd.DataFrame:
    """Return per-class counts and percentages for a patient-level frame."""
    counts = frame["label"].value_counts().sort_index()
    distribution = pd.DataFrame(
        {
            "label": counts.index.astype(int),
            "class_name": [CLASS_NAMES[int(label)] for label in counts.index],
            "count": counts.to_numpy(dtype=int),
        }
    )
    distribution["percentage"] = 100.0 * distribution["count"] / float(len(frame))
    return distribution


def imbalance_summary(frame: pd.DataFrame) -> dict[str, float]:
    """Compute positive prevalence and the negative/positive imbalance ratio."""
    n_total = int(len(frame))
    n_positive = int((frame["label"] == 1).sum())
    n_negative = int((frame["label"] == 0).sum())
    prevalence = n_positive / n_total if n_total else float("nan")
    ratio = n_negative / n_positive if n_positive else float("inf")
    return {
        "n_total": n_total,
        "n_negative": n_negative,
        "n_positive": n_positive,
        "positive_prevalence": prevalence,
        "imbalance_ratio_neg_over_pos": ratio,
    }


def plot_class_distribution(frame: pd.DataFrame, title: str = "Patient-level class distribution") -> None:
    """Plot the class distribution as a labelled bar chart."""
    distribution = class_distribution(frame)
    figure, axis = plt.subplots(figsize=(6, 4))
    bars = axis.bar(distribution["class_name"], distribution["count"], color=["#4C72B0", "#C44E52"])
    for bar, percentage in zip(bars, distribution["percentage"]):
        axis.annotate(
            f"{int(bar.get_height())}\n({percentage:.1f}%)",
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            ha="center",
            va="bottom",
            fontsize=10,
        )
    axis.set_xlabel("Class")
    axis.set_ylabel("Number of patients")
    axis.set_title(title)
    axis.margins(y=0.15)
    figure.tight_layout()
    plt.show()


def describe_dicom_headers(frame: pd.DataFrame, n_per_class: int = 3, seed: int = SEED) -> pd.DataFrame:
    """Report image dimensions and photometric metadata for a reproducible sample."""
    sample = sample_per_class(frame, n_per_class=n_per_class, seed=seed)
    records: list[dict[str, Any]] = []
    for row in sample.itertuples(index=False):
        dataset = pydicom.dcmread(Path(row.dicom_path), stop_before_pixels=True)
        records.append(
            {
                "patientId": row.patientId,
                "class_name": row.class_name,
                "rows": getattr(dataset, "Rows", None),
                "columns": getattr(dataset, "Columns", None),
                "photometric_interpretation": getattr(dataset, "PhotometricInterpretation", None),
                "samples_per_pixel": getattr(dataset, "SamplesPerPixel", None),
                "bits_stored": getattr(dataset, "BitsStored", None),
                "pixel_representation": getattr(dataset, "PixelRepresentation", None),
                "modality": getattr(dataset, "Modality", None),
                "body_part": getattr(dataset, "BodyPartExamined", None),
                "view_position": getattr(dataset, "ViewPosition", None),
            }
        )
    return pd.DataFrame.from_records(records)


def preview_class_examples(
    frame: pd.DataFrame,
    decode_fn: Callable[[Path], Image.Image],
    n_per_class: int = 3,
    seed: int = SEED,
) -> None:
    """Show a reproducible grid of decoded DICOM examples from both classes.

    ``decode_fn`` is injected so this exploratory helper reuses the single canonical
    decoder defined in section 7 instead of duplicating the decoding logic.
    """
    sample = sample_per_class(frame, n_per_class=n_per_class, seed=seed)
    n_images = len(sample)
    columns = min(n_images, 2 * n_per_class)
    rows = int(np.ceil(n_images / columns)) if columns else 1
    figure, axes = plt.subplots(rows, columns, figsize=(3.0 * columns, 3.2 * rows))
    axes_list = np.atleast_1d(axes).ravel()
    for axis, row in zip(axes_list, sample.itertuples(index=False)):
        image = decode_fn(Path(row.dicom_path))
        axis.imshow(np.asarray(image))
        axis.set_title(f"{row.class_name}\n{row.patientId[:8]}...", fontsize=9)
        axis.axis("off")
    for axis in axes_list[n_images:]:
        axis.axis("off")
    figure.suptitle("Decoded DICOM examples per class", fontsize=12)
    figure.tight_layout()
    plt.show()

In [ ]:
"""Run the exploratory analysis on the aggregated patient frame."""

print(f"Unique patients: {patients_df['patientId'].nunique()}")
print()
print("Class counts and percentages")
print(class_distribution(patients_df).to_string(index=False))
print()

imbalance = imbalance_summary(patients_df)
print(f"Positive prevalence          : {imbalance['positive_prevalence']:.4f} "
      f"({100.0 * imbalance['positive_prevalence']:.2f}%)")
print(f"Imbalance ratio (neg / pos)  : {imbalance['imbalance_ratio_neg_over_pos']:.4f}")

plot_class_distribution(patients_df)

In [ ]:
"""Inspect dimensions and photometric metadata for a reproducible sample of files."""

header_sample = describe_dicom_headers(patients_df, n_per_class=3, seed=SEED)
print("Sampled DICOM header information")
print(header_sample.to_string(index=False))
print()
print("Distinct photometric interpretations in the sample:",
      sorted(set(header_sample["photometric_interpretation"].dropna())))
sampled_sizes = {
    (int(rows), int(columns))
    for rows, columns in zip(header_sample["rows"], header_sample["columns"])
    if rows is not None and columns is not None
}
print("Distinct image sizes in the sample:", sorted(sampled_sizes))

---
## 6. Patient-level stratified 70/30 split

The split is performed **once, at patient level** (one image per patient after aggregation),
stratified on the aggregated binary target with `random_state=42`. No image is ever copied
into class folders: the dataset reads DICOM files directly from the dataframe.

Assertions guarantee that the two patient-ID sets are disjoint, that their union covers the
whole cohort, and that each patient appears in exactly one split.

In [ ]:
"""Reproducible stratified patient-level split with explicit disjointness checks."""


def stratified_patient_split(
    frame: pd.DataFrame,
    train_fraction: float = TRAIN_FRACTION,
    seed: int = SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split patients into train/validation, stratified on the aggregated binary label."""
    train_frame, validation_frame = train_test_split(
        frame,
        train_size=train_fraction,
        stratify=frame["label"],
        random_state=seed,
        shuffle=True,
    )
    train_frame = train_frame.reset_index(drop=True)
    validation_frame = validation_frame.reset_index(drop=True)

    train_ids = set(train_frame["patientId"])
    validation_ids = set(validation_frame["patientId"])
    assert not (train_ids & validation_ids), "Train and validation patient IDs overlap."
    assert train_ids | validation_ids == set(frame["patientId"]), (
        "The split does not cover every patient exactly once."
    )
    assert len(train_frame) + len(validation_frame) == len(frame), (
        "Split sizes do not add up to the cohort size."
    )
    assert train_frame["patientId"].is_unique and validation_frame["patientId"].is_unique, (
        "A patient appears more than once inside a split."
    )
    for name, split_frame in (("train", train_frame), ("validation", validation_frame)):
        present = set(split_frame["label"].unique())
        assert present == {0, 1}, f"The {name} split must contain both classes, found {present}."
    return train_frame, validation_frame


def save_split(frame: pd.DataFrame, filename: str) -> Path:
    """Persist patient IDs and labels of one split inside ARTIFACT_DIR."""
    path = safe_artifact_path(filename)
    frame[["patientId", "label", "class_name"]].to_csv(path, index=False)
    return path


train_df, validation_df = stratified_patient_split(patients_df, TRAIN_FRACTION, SEED)

train_split_path = save_split(train_df, TRAIN_SPLIT_FILENAME)
validation_split_path = save_split(validation_df, VALIDATION_SPLIT_FILENAME)

print(f"Train patients      : {len(train_df)} ({100.0 * len(train_df) / len(patients_df):.2f}%)")
print(f"Validation patients : {len(validation_df)} "
      f"({100.0 * len(validation_df) / len(patients_df):.2f}%)")
print()
print("Train class distribution")
print(class_distribution(train_df).to_string(index=False))
print()
print("Validation class distribution")
print(class_distribution(validation_df).to_string(index=False))
print()
print(f"Saved: {train_split_path}")
print(f"Saved: {validation_split_path}")

---
## 7. DICOM decoding and preprocessing

A single, reusable decoder converts a chest-radiograph DICOM into an 8-bit RGB `PIL.Image`:

1. read the pixel array with `pydicom`;
2. apply the VOI LUT when the file provides one;
3. invert `MONOCHROME1` images so that higher values always mean brighter tissue;
4. replace non-finite values safely;
5. clip intensities robustly using image percentiles (`0.5` / `99.5`);
6. rescale to `[0, 255]` and build a grayscale image;
7. replicate the single channel three times, because ImageNet-pretrained ResNet50 expects RGB.

Unreadable or structurally invalid files raise `DicomDecodeError` with a clear message.

In [ ]:
"""Single canonical DICOM decoder shared by EDA, training, validation and the smoke test."""


class DicomDecodeError(RuntimeError):
    """Raised when a DICOM file cannot be read or does not contain usable pixel data."""


def read_dicom_array(path: Path) -> np.ndarray:
    """Read a DICOM file and return a finite float32 array with photometric correction."""
    try:
        dataset = pydicom.dcmread(str(path))
        pixels = dataset.pixel_array
    except Exception as exc:  # noqa: BLE001 - surface every failure as one clear error
        raise DicomDecodeError(f"Could not read DICOM file '{path}': {exc}") from exc

    if pixels is None or getattr(pixels, "size", 0) == 0:
        raise DicomDecodeError(f"DICOM file '{path}' contains no pixel data.")

    try:
        pixels = apply_voi_lut(pixels, dataset)
    except Exception:  # noqa: BLE001 - a missing or malformed LUT must not abort decoding
        pass

    array = np.asarray(pixels, dtype=np.float32)
    if array.ndim == 3:  # collapse an unexpected channel axis to grayscale
        array = array.mean(axis=-1, dtype=np.float32)
    if array.ndim != 2:
        raise DicomDecodeError(f"DICOM file '{path}' has an unsupported shape {array.shape}.")

    finite_mask = np.isfinite(array)
    if not finite_mask.any():
        raise DicomDecodeError(f"DICOM file '{path}' contains no finite pixel values.")
    if not finite_mask.all():
        array = np.where(finite_mask, array, np.median(array[finite_mask])).astype(np.float32)

    if str(getattr(dataset, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
        array = float(array.max()) - array

    return array


def scale_to_uint8(array: np.ndarray, percentiles: tuple[float, float] = CLIP_PERCENTILES) -> np.ndarray:
    """Robustly clip with image percentiles and rescale intensities to [0, 255]."""
    low, high = np.percentile(array, percentiles)
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low, high = float(array.min()), float(array.max())
    if high <= low:  # constant image
        return np.zeros(array.shape, dtype=np.uint8)
    clipped = np.clip(array, low, high)
    scaled = (clipped - low) / (high - low) * 255.0
    return scaled.round().astype(np.uint8)


def dicom_to_rgb_image(path: Path) -> Image.Image:
    """Decode a DICOM file into an 8-bit RGB PIL image with three identical channels."""
    array = read_dicom_array(Path(path))
    grayscale = Image.fromarray(scale_to_uint8(array), mode="L")
    rgb_image = grayscale.convert("RGB")
    assert rgb_image.mode == "RGB", "The decoder must return a 3-channel RGB image."
    return rgb_image

In [ ]:
"""Exploratory grid of decoded examples (uses the canonical decoder defined above)."""

preview_class_examples(patients_df, decode_fn=dicom_to_rgb_image, n_per_class=3, seed=SEED)

---
## 8. Data augmentation and DataLoaders

Augmentation is applied **only to the training dataset** and stays medically plausible for
frontal chest radiographs: a small rotation (±7°), a translation of at most 5%, a mild scale
change, an optional horizontal flip and a mild brightness/contrast jitter. Vertical flips,
90°/180° rotations and aggressive crops are deliberately excluded — they either produce
anatomically impossible images or remove lung regions that carry the diagnostic signal.

Resizing to `224 × 224` keeps the whole thoracic field (no cropping).

**Validation preprocessing is fully deterministic**: resize, tensor conversion and ImageNet
normalisation only. The very same evaluation transform is reused later for deployment
inference, so a single definition covers validation and production.

In [ ]:
"""Transforms, Dataset and DataLoaders."""


def build_train_transforms() -> transforms.Compose:
    """Conservative, medically plausible augmentation for the training split only."""
    return transforms.Compose(
        [
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomAffine(
                degrees=ROTATION_DEGREES,
                translate=(TRANSLATE_FRACTION, TRANSLATE_FRACTION),
                scale=SCALE_RANGE,
                interpolation=transforms.InterpolationMode.BILINEAR,
                fill=0,
            ),
            transforms.RandomHorizontalFlip(p=HORIZONTAL_FLIP_P),
            transforms.ColorJitter(brightness=BRIGHTNESS_JITTER, contrast=CONTRAST_JITTER),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )


def build_eval_transforms() -> transforms.Compose:
    """Fully deterministic preprocessing used for validation and for deployment."""
    return transforms.Compose(
        [
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )


TRAIN_TRANSFORMS = build_train_transforms()
EVAL_TRANSFORMS = build_eval_transforms()


class RSNAPneumoniaDataset(Dataset):
    """Patient-level dataset that decodes DICOM files directly from a dataframe."""

    REQUIRED_COLUMNS = ("patientId", "label", "dicom_path")

    def __init__(self, frame: pd.DataFrame, transform: Callable[[Image.Image], torch.Tensor]) -> None:
        missing = [column for column in self.REQUIRED_COLUMNS if column not in frame.columns]
        if missing:
            raise ValueError(f"Dataset frame is missing columns: {missing}")
        self.records = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.records.iloc[index]
        label = int(row["label"])
        assert label in (0, 1), f"Invalid label {label} for patient {row['patientId']}."
        image = dicom_to_rgb_image(Path(row["dicom_path"]))
        tensor = self.transform(image)
        assert tensor.shape == (3, IMAGE_SIZE, IMAGE_SIZE), (
            f"Expected a (3, {IMAGE_SIZE}, {IMAGE_SIZE}) tensor, got {tuple(tensor.shape)}."
        )
        return tensor, torch.tensor(float(label), dtype=torch.float32)


def build_dataloader(dataset: Dataset, shuffle: bool, seed: int = SEED) -> DataLoader:
    """Create a reproducible DataLoader; persistent workers only when workers exist."""
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=NUM_WORKERS > 0,
    )


train_dataset = RSNAPneumoniaDataset(train_df, TRAIN_TRANSFORMS)
validation_dataset = RSNAPneumoniaDataset(validation_df, EVAL_TRANSFORMS)

train_loader = build_dataloader(train_dataset, shuffle=True)
validation_loader = build_dataloader(validation_dataset, shuffle=False)

print(f"Train samples      : {len(train_dataset)} in {len(train_loader)} batches (shuffled)")
print(f"Validation samples : {len(validation_dataset)} in {len(validation_loader)} batches (not shuffled)")

In [ ]:
"""Display one augmented training batch with denormalised images."""


def show_augmented_batch(loader: DataLoader, n_images: int = 8) -> None:
    """Plot a denormalised grid from a single batch of the given loader."""
    images, labels = next(iter(loader))
    assert images.ndim == 4 and images.shape[1] == 3, f"Unexpected batch shape {tuple(images.shape)}."
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    denormalised = (images * std + mean).clamp(0.0, 1.0)

    count = min(n_images, denormalised.shape[0])
    columns = min(4, count)
    rows = int(np.ceil(count / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(3.0 * columns, 3.2 * rows))
    axes_list = np.atleast_1d(axes).ravel()
    for position in range(count):
        axes_list[position].imshow(denormalised[position].permute(1, 2, 0).numpy())
        axes_list[position].set_title(CLASS_NAMES[int(labels[position].item())], fontsize=9)
        axes_list[position].axis("off")
    for axis in axes_list[count:]:
        axis.axis("off")
    figure.suptitle("Augmented training batch (denormalised)", fontsize=12)
    figure.tight_layout()
    plt.show()


show_augmented_batch(train_loader, n_images=8)

---
## 9. Class-imbalance handling

The imbalance statistics come **exclusively from the training split**. The validation split
is never oversampled, never augmented and never used to derive weights.

`BCEWithLogitsLoss` receives

```
pos_weight = number_of_negative_training_samples / number_of_positive_training_samples
```

so a single positive sample counts as much as `pos_weight` negatives when computing the loss.

In [ ]:
"""Positive-class weighting derived from the training split only."""


def compute_pos_weight(frame: pd.DataFrame) -> float:
    """Return negatives / positives for the given (training) frame."""
    n_positive = int((frame["label"] == 1).sum())
    n_negative = int((frame["label"] == 0).sum())
    if n_positive == 0:
        raise ValueError("The training split contains no positive sample; pos_weight is undefined.")
    return n_negative / n_positive


def build_criterion(pos_weight_value: float, device: torch.device) -> nn.BCEWithLogitsLoss:
    """Build the binary loss; the model must emit raw logits, never sigmoid outputs."""
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)


train_imbalance = imbalance_summary(train_df)
POS_WEIGHT = compute_pos_weight(train_df)
criterion = build_criterion(POS_WEIGHT, DEVICE)

print("Training split only")
print(f"  negatives ({CLASS_NAMES[0]}): {train_imbalance['n_negative']}")
print(f"  positives ({CLASS_NAMES[1]}): {train_imbalance['n_positive']}")
print(f"  positive prevalence         : {train_imbalance['positive_prevalence']:.4f}")
print(f"  pos_weight (neg / pos)      : {POS_WEIGHT:.4f}")
print(f"  loss                        : {type(criterion).__name__} with pos_weight")

---
## 10. ResNet50 definition

`torchvision.models.resnet50(weights=ResNet50_Weights.DEFAULT)` provides the ImageNet
initialisation. The 1000-way classifier is replaced by a binary head made of a dropout layer
and a single linear output logit; `forward` returns a tensor of shape `[batch_size]`, which is
exactly what `BCEWithLogitsLoss` expects. No sigmoid is applied inside the model.

The whole backbone is fine-tuned (nothing is permanently frozen), with **AdamW** and a
`ReduceLROnPlateau` scheduler driven by the validation loss.

In [ ]:
"""Binary ResNet50 classifier and optimisation objects."""


class PneumoniaResNet50(nn.Module):
    """ImageNet-pretrained ResNet50 with a single-logit binary head."""

    def __init__(self, dropout: float = DROPOUT, pretrained: bool = True) -> None:
        super().__init__()
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone = resnet50(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 1),
        )
        for parameter in self.backbone.parameters():  # full fine-tuning, nothing frozen
            parameter.requires_grad = True

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Return raw logits with shape ``[batch_size]`` (no sigmoid)."""
        logits = self.backbone(pixel_values)
        assert logits.shape[1] == 1, f"Expected one output logit, got {tuple(logits.shape)}."
        return logits.flatten(start_dim=0)


def build_model(dropout: float = DROPOUT, pretrained: bool = True) -> PneumoniaResNet50:
    """Instantiate the classifier used for training and export."""
    return PneumoniaResNet50(dropout=dropout, pretrained=pretrained)


def count_trainable_parameters(model: nn.Module) -> int:
    """Count parameters that receive gradients."""
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


HYPERPARAMETERS: dict[str, Any] = {
    "architecture": ARCHITECTURE,
    "max_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT,
    "decision_threshold": DECISION_THRESHOLD,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "optimizer": "AdamW",
    "scheduler": "ReduceLROnPlateau(monitor=validation_loss)",
    "loss": "BCEWithLogitsLoss(pos_weight=negatives/positives)",
    "pos_weight": POS_WEIGHT,
    "seed": SEED,
    "train_fraction": TRAIN_FRACTION,
    "validation_fraction": VALIDATION_FRACTION,
    "fine_tuning": "full backbone",
}

set_seed(SEED)
model = build_model(dropout=DROPOUT, pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
)

with torch.no_grad():
    _probe = model(torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE))
assert _probe.shape == (2,), f"forward() must return shape [batch_size], got {tuple(_probe.shape)}."
del _probe

print(f"Model               : {ARCHITECTURE} + Dropout({DROPOUT}) + Linear(->1)")
print(f"Trainable parameters: {count_trainable_parameters(model):,}")
print(f"Optimizer           : AdamW(lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler           : ReduceLROnPlateau(mode='min', factor={SCHEDULER_FACTOR}, "
      f"patience={SCHEDULER_PATIENCE})")

---
## 11. Training and validation functions

Both functions are reusable and free of hidden global state — every dependency is passed in
explicitly. Key properties:

- automatic CPU/CUDA device handling, mixed precision only on CUDA, version-safe AMP;
- gradients enabled only in `train_one_epoch`; `validate_one_epoch` runs under
  `torch.inference_mode()`;
- `model.train()` / `model.eval()` toggled correctly;
- the epoch loss is a **sample-weighted** mean (`sum(loss * batch_size) / n_samples`), not an
  unweighted average of batch averages;
- the sigmoid is applied **only** to obtain probabilities, never before `BCEWithLogitsLoss`;
- validation probabilities and labels are moved to CPU as NumPy arrays batch by batch, so no
  unnecessary tensor is retained on the GPU.

In [ ]:
"""Reusable single-epoch training and validation routines."""


@dataclass
class ValidationOutputs:
    """Validation results for one epoch."""

    loss: float
    accuracy: float
    probabilities: np.ndarray
    labels: np.ndarray


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    scaler: Any | None = None,
    use_amp: bool = False,
    threshold: float = DECISION_THRESHOLD,
) -> tuple[float, float]:
    """Run one training epoch and return the sample-weighted loss and accuracy."""
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        assert labels.ndim == 1, f"Labels must be 1-D, got {tuple(labels.shape)}."

        optimizer.zero_grad(set_to_none=True)
        with amp_autocast(device.type, enabled=use_amp):
            logits = model(images)
            assert logits.shape == labels.shape, (
                f"Logits {tuple(logits.shape)} and labels {tuple(labels.shape)} must match."
            )
            loss = criterion(logits, labels)  # raw logits go into BCEWithLogitsLoss

        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        batch_size = int(labels.size(0))
        total_loss += float(loss.item()) * batch_size
        with torch.no_grad():
            predictions = (torch.sigmoid(logits.detach().float()) >= threshold).to(labels.dtype)
            total_correct += int((predictions == labels).sum().item())
        total_samples += batch_size

    if total_samples == 0:
        raise RuntimeError("The training loader produced no sample.")
    return total_loss / total_samples, total_correct / total_samples


def validate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    threshold: float = DECISION_THRESHOLD,
    use_amp: bool = False,
) -> ValidationOutputs:
    """Evaluate the model and collect probabilities and labels as CPU NumPy arrays."""
    model.eval()
    total_loss = 0.0
    total_samples = 0
    probability_chunks: list[np.ndarray] = []
    label_chunks: list[np.ndarray] = []

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with amp_autocast(device.type, enabled=use_amp):
                logits = model(images)
                loss = criterion(logits, labels)

            batch_size = int(labels.size(0))
            total_loss += float(loss.item()) * batch_size
            total_samples += batch_size
            probability_chunks.append(torch.sigmoid(logits.float()).cpu().numpy())
            label_chunks.append(labels.float().cpu().numpy())

    if total_samples == 0:
        raise RuntimeError("The validation loader produced no sample.")

    probabilities = np.concatenate(probability_chunks).astype(np.float64)
    labels_array = np.concatenate(label_chunks).astype(np.int64)
    assert probabilities.shape == labels_array.shape, "Probabilities and labels must align."
    assert set(np.unique(labels_array)).issubset({0, 1}), "Validation labels must be 0 or 1."

    predictions = (probabilities >= threshold).astype(np.int64)
    return ValidationOutputs(
        loss=total_loss / total_samples,
        accuracy=float((predictions == labels_array).mean()),
        probabilities=probabilities,
        labels=labels_array,
    )

---
## 12. Training with early stopping

At most **100 epochs**. Early stopping monitors **the validation loss only**:

- a checkpoint is written whenever the validation loss improves by at least `1e-4`;
- the patience counter resets after every improvement;
- training stops after **10** consecutive epochs without a sufficient improvement;
- the checkpoint always holds the **best** epoch, not the last one, and it is reloaded before
  the final evaluation.

> The training call is present but has **not** been executed. Run it after configuring the
> dataset paths; expect a long runtime on the full cohort.

In [ ]:
"""Training loop with validation-loss early stopping and best-epoch checkpointing."""


@dataclass
class TrainingHistory:
    """Per-epoch training and validation history."""

    epochs: list[int] = field(default_factory=list)
    learning_rates: list[float] = field(default_factory=list)
    train_losses: list[float] = field(default_factory=list)
    train_accuracies: list[float] = field(default_factory=list)
    val_losses: list[float] = field(default_factory=list)
    val_accuracies: list[float] = field(default_factory=list)
    epoch_seconds: list[float] = field(default_factory=list)

    def append(
        self,
        epoch: int,
        learning_rate: float,
        train_loss: float,
        train_accuracy: float,
        val_loss: float,
        val_accuracy: float,
        elapsed: float,
    ) -> None:
        """Record one epoch."""
        self.epochs.append(epoch)
        self.learning_rates.append(learning_rate)
        self.train_losses.append(train_loss)
        self.train_accuracies.append(train_accuracy)
        self.val_losses.append(val_loss)
        self.val_accuracies.append(val_accuracy)
        self.epoch_seconds.append(elapsed)

    def to_frame(self) -> pd.DataFrame:
        """Return the history as a dataframe."""
        return pd.DataFrame(
            {
                "epoch": self.epochs,
                "learning_rate": self.learning_rates,
                "train_loss": self.train_losses,
                "train_accuracy": self.train_accuracies,
                "val_loss": self.val_losses,
                "val_accuracy": self.val_accuracies,
                "seconds": self.epoch_seconds,
            }
        )


def fit(
    model: nn.Module,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: Any,
    device: torch.device,
    checkpoint_path: Path,
    max_epochs: int = MAX_EPOCHS,
    patience: int = EARLY_STOPPING_PATIENCE,
    min_delta: float = EARLY_STOPPING_MIN_DELTA,
    threshold: float = DECISION_THRESHOLD,
    use_amp: bool = USE_AMP,
) -> tuple[TrainingHistory, dict[str, Any]]:
    """Train with early stopping on the validation loss and keep the best checkpoint."""
    history = TrainingHistory()
    scaler = make_grad_scaler(use_amp)
    best_val_loss = float("inf")
    best_epoch = -1
    epochs_without_improvement = 0
    last_epoch = 0

    for epoch in range(1, max_epochs + 1):
        last_epoch = epoch
        started = time.perf_counter()
        current_lr = float(optimizer.param_groups[0]["lr"])

        train_loss, train_accuracy = train_one_epoch(
            model, train_loader, criterion, optimizer, device,
            scaler=scaler, use_amp=use_amp, threshold=threshold,
        )
        validation = validate_one_epoch(
            model, validation_loader, criterion, device,
            threshold=threshold, use_amp=use_amp,
        )
        scheduler.step(validation.loss)
        elapsed = time.perf_counter() - started

        history.append(
            epoch=epoch,
            learning_rate=current_lr,
            train_loss=train_loss,
            train_accuracy=train_accuracy,
            val_loss=validation.loss,
            val_accuracy=validation.accuracy,
            elapsed=elapsed,
        )
        print(
            f"Epoch {epoch:3d}/{max_epochs} | lr {current_lr:.2e} | "
            f"train loss {train_loss:.4f} acc {train_accuracy:.4f} | "
            f"val loss {validation.loss:.4f} acc {validation.accuracy:.4f} | "
            f"{elapsed:.1f}s"
        )

        if best_val_loss - validation.loss >= min_delta:
            best_val_loss = validation.loss
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "epoch": epoch,
                    "val_loss": validation.loss,
                    "val_accuracy": validation.accuracy,
                    "architecture": ARCHITECTURE,
                    "model_state_dict": model.state_dict(),
                },
                checkpoint_path,
            )
            print(f"    validation loss improved -> checkpoint saved (epoch {epoch})")
        else:
            epochs_without_improvement += 1
            print(
                f"    no improvement >= {min_delta} "
                f"({epochs_without_improvement}/{patience})"
            )
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
                break

    if best_epoch < 0:
        raise RuntimeError("No checkpoint was saved: the validation loss never improved.")

    summary = {
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "epochs_completed": last_epoch,
        "max_epochs": max_epochs,
        "early_stopping_patience": patience,
        "early_stopping_min_delta": min_delta,
        "early_stopped": last_epoch < max_epochs,
        "checkpoint_path": str(checkpoint_path),
    }
    return history, summary


def load_best_checkpoint(model: nn.Module, checkpoint_path: Path, device: torch.device) -> dict[str, Any]:
    """Reload the best checkpoint into ``model`` and return its metadata."""
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    except TypeError:  # torch < 2.0 has no weights_only argument
        checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    return checkpoint

In [ ]:
"""Run training (long-running; not executed while the notebook is generated)."""

BEST_CHECKPOINT_PATH = safe_artifact_path(BEST_CHECKPOINT_FILENAME)

set_seed(SEED)
history, training_summary = fit(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    checkpoint_path=BEST_CHECKPOINT_PATH,
    max_epochs=MAX_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    min_delta=EARLY_STOPPING_MIN_DELTA,
    threshold=DECISION_THRESHOLD,
    use_amp=USE_AMP,
)

print()
for key, value in training_summary.items():
    print(f"{key:>26}: {value}")

---
## 13. Training-history plots

Loss and accuracy curves for both splits, with the best epoch highlighted. All values come
from the `TrainingHistory` object produced by the run above — nothing is transcribed by hand.

In [ ]:
"""Plot the training history."""


def plot_training_history(history: TrainingHistory, best_epoch: int | None = None) -> None:
    """Plot train/validation loss and accuracy curves side by side."""
    if not history.epochs:
        print("No epoch was recorded; run the training cell first.")
        return

    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(history.epochs, history.train_losses, label="Training loss")
    axes[0].plot(history.epochs, history.val_losses, label="Validation loss")
    axes[0].set_title("Training and validation loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(history.epochs, history.train_accuracies, label="Training accuracy")
    axes[1].plot(history.epochs, history.val_accuracies, label="Validation accuracy")
    axes[1].set_title("Training and validation accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    if best_epoch is not None and best_epoch > 0:
        for axis in axes:
            axis.axvline(best_epoch, color="grey", linestyle="--", linewidth=1,
                         label=f"Best epoch ({best_epoch})")
        axes[0].legend()
        axes[1].legend()

    figure.tight_layout()
    plt.show()


def plot_learning_rate(history: TrainingHistory) -> None:
    """Plot the learning-rate schedule actually applied during training."""
    if not history.epochs:
        print("No epoch was recorded; run the training cell first.")
        return
    figure, axis = plt.subplots(figsize=(6.5, 4))
    axis.plot(history.epochs, history.learning_rates, marker="o", markersize=3)
    axis.set_title("Learning rate per epoch")
    axis.set_xlabel("Epoch")
    axis.set_ylabel("Learning rate")
    axis.set_yscale("log")
    axis.grid(alpha=0.3)
    figure.tight_layout()
    plt.show()


plot_training_history(history, best_epoch=training_summary["best_epoch"])
plot_learning_rate(history)
history.to_frame().tail(10)

---
## 14. Final validation evaluation

The best checkpoint is reloaded and evaluated on the untouched validation split at the
decision threshold `0.50`, in full precision. Metrics that are mathematically undefined
(for example precision when the model predicts no positive at all, or ROC-AUC when a class
is absent) are returned as `None` together with an explicit warning instead of a misleading
zero.

No metric value is written by hand anywhere in this notebook: every number below is produced
by the code at runtime.

In [ ]:
"""Binary metric computation with explicit handling of undefined values."""


def compute_binary_metrics(
    y_true: np.ndarray,
    y_probabilities: np.ndarray,
    threshold: float = DECISION_THRESHOLD,
) -> dict[str, Any]:
    """Compute threshold and ranking metrics, returning None for undefined quantities."""
    y_true = np.asarray(y_true).astype(int).ravel()
    y_probabilities = np.asarray(y_probabilities).astype(float).ravel()
    assert y_true.shape == y_probabilities.shape, "Labels and probabilities must align."
    assert set(np.unique(y_true)).issubset({0, 1}), "Labels must be binary."

    y_pred = (y_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tn, fp, fn, tp = int(tn), int(fp), int(fn), int(tp)

    warnings: list[str] = []

    def _ratio(numerator: int, denominator: int, name: str) -> float | None:
        if denominator == 0:
            warnings.append(f"{name} is undefined (denominator is zero).")
            return None
        return numerator / denominator

    precision = _ratio(tp, tp + fp, "precision")
    recall = _ratio(tp, tp + fn, "recall/sensitivity")
    specificity = _ratio(tn, tn + fp, "specificity")

    if precision is None or recall is None or (precision + recall) == 0:
        if precision is not None and recall is not None:
            warnings.append("F1 is undefined (precision + recall = 0).")
        f1 = None
    else:
        f1 = 2.0 * precision * recall / (precision + recall)

    if recall is None or specificity is None:
        warnings.append("Balanced accuracy is undefined (sensitivity or specificity missing).")
        balanced_accuracy = None
    else:
        balanced_accuracy = 0.5 * (recall + specificity)

    if len(np.unique(y_true)) < 2:
        warnings.append("ROC-AUC and PR-AUC are undefined: the validation split has a single class.")
        roc_auc = None
        pr_auc = None
    else:
        roc_auc = float(roc_auc_score(y_true, y_probabilities))
        pr_auc = float(average_precision_score(y_true, y_probabilities))

    return {
        "threshold": float(threshold),
        "n_samples": int(y_true.size),
        "support_negative": int((y_true == 0).sum()),
        "support_positive": int((y_true == 1).sum()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall_sensitivity": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc_average_precision": pr_auc,
        "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp},
        "undefined_metrics": warnings,
    }


def format_metric(value: float | None) -> str:
    """Render a metric value, making undefined values explicit."""
    return "undefined" if value is None else f"{value:.4f}"


def plot_confusion_matrix(counts: dict[str, int]) -> None:
    """Plot the 2x2 confusion matrix with absolute counts."""
    matrix = np.array([[counts["tn"], counts["fp"]], [counts["fn"], counts["tp"]]], dtype=int)
    figure, axis = plt.subplots(figsize=(5, 4.5))
    image = axis.imshow(matrix, cmap="Blues")
    axis.set_xticks([0, 1], labels=[CLASS_NAMES[0], CLASS_NAMES[1]])
    axis.set_yticks([0, 1], labels=[CLASS_NAMES[0], CLASS_NAMES[1]])
    axis.set_xlabel("Predicted label")
    axis.set_ylabel("True label")
    axis.set_title("Validation confusion matrix")
    for row in range(2):
        for column in range(2):
            axis.text(column, row, f"{matrix[row, column]}", ha="center", va="center",
                      color="black", fontsize=12)
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    figure.tight_layout()
    plt.show()


def plot_roc_and_pr_curves(y_true: np.ndarray, y_probabilities: np.ndarray,
                           roc_auc: float | None, pr_auc: float | None) -> None:
    """Plot the ROC and precision-recall curves when both classes are present."""
    if len(np.unique(y_true)) < 2:
        print("ROC and PR curves are undefined: the validation split has a single class.")
        return
    false_positive_rate, true_positive_rate, _ = roc_curve(y_true, y_probabilities)
    precision_values, recall_values, _ = precision_recall_curve(y_true, y_probabilities)
    positive_rate = float(np.mean(y_true))

    figure, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    axes[0].plot(false_positive_rate, true_positive_rate,
                 label=f"ROC (AUC = {format_metric(roc_auc)})")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey", label="Chance")
    axes[0].set_title("ROC curve - validation split")
    axes[0].set_xlabel("False positive rate (1 - specificity)")
    axes[0].set_ylabel("True positive rate (sensitivity)")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(recall_values, precision_values,
                 label=f"PR (AP = {format_metric(pr_auc)})")
    axes[1].axhline(positive_rate, linestyle="--", color="grey",
                    label=f"Positive rate ({positive_rate:.3f})")
    axes[1].set_title("Precision-recall curve - validation split")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    figure.tight_layout()
    plt.show()

In [ ]:
"""Reload the best checkpoint and evaluate it on the untouched validation split."""

best_checkpoint = load_best_checkpoint(model, BEST_CHECKPOINT_PATH, DEVICE)
print(f"Reloaded checkpoint from epoch {best_checkpoint['epoch']} "
      f"(validation loss {best_checkpoint['val_loss']:.6f})")

final_validation = validate_one_epoch(
    model=model,
    loader=validation_loader,
    criterion=criterion,
    device=DEVICE,
    threshold=DECISION_THRESHOLD,
    use_amp=False,  # full precision for the reported evaluation
)
validation_metrics = compute_binary_metrics(
    final_validation.labels, final_validation.probabilities, DECISION_THRESHOLD
)
validation_metrics["validation_loss"] = float(final_validation.loss)

confusion_counts = validation_metrics["confusion_matrix"]
print()
print("Validation results at threshold "
      f"{validation_metrics['threshold']:.2f} (best epoch {best_checkpoint['epoch']})")
print("-" * 64)
print(f"{'validation loss':>26}: {validation_metrics['validation_loss']:.6f}")
for key in (
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1",
    "roc_auc",
    "pr_auc_average_precision",
):
    print(f"{key:>26}: {format_metric(validation_metrics[key])}")
print(f"{'confusion matrix':>26}: TN={confusion_counts['tn']} FP={confusion_counts['fp']} "
      f"FN={confusion_counts['fn']} TP={confusion_counts['tp']}")
print(f"{'support (neg / pos)':>26}: {validation_metrics['support_negative']} / "
      f"{validation_metrics['support_positive']}")
for warning in validation_metrics["undefined_metrics"]:
    print(f"WARNING: {warning}")

print()
print("Classification report")
print(
    classification_report(
        final_validation.labels,
        (final_validation.probabilities >= DECISION_THRESHOLD).astype(int),
        labels=[0, 1],
        target_names=[CLASS_NAMES[0], CLASS_NAMES[1]],
        digits=4,
        zero_division=0,
    )
)

plot_confusion_matrix(confusion_counts)
plot_roc_and_pr_curves(
    final_validation.labels,
    final_validation.probabilities,
    validation_metrics["roc_auc"],
    validation_metrics["pr_auc_average_precision"],
)
plot_training_history(history, best_epoch=training_summary["best_epoch"])

---
## 15. ED²A artifact export

The ED²A application already ships the DDXPlus differential-diagnosis artifacts
`best_model.pkl`, `preprocessor.pkl`, `label_encoder.pkl`, `model_metrics.json` and
`artifact_manifest.json` in `API_App/artifacts/`. **Those names belong to another model and
are never produced here**: `safe_artifact_path()` raises on every one of them, and every file
written by this notebook is prefixed with `advanced_pneumonia_`.

The bundle exported below contains:

| File | Purpose |
| --- | --- |
| `advanced_pneumonia_checkpoint.pt` | best `state_dict` plus architecture, best epoch, best validation loss, class mapping, threshold, normalisation, input size and hyperparameters |
| `advanced_pneumonia_model.onnx` | logit-producing graph, dynamic batch, validated with `onnx.checker` |
| `advanced_pneumonia_config.json` | complete inference contract for the future service |
| `advanced_pneumonia_metrics.json` | real post-training validation metrics |
| `advanced_pneumonia_manifest.json` | timestamp, versions, sizes and SHA-256 checksums |
| `advanced_pneumonia_requirements.txt` | minimal runtime dependencies |
| `ed2a_advanced_pneumonia_artifacts.zip` | the five deployable files in one archive |

In [ ]:
"""Export functions for the ED2A advanced_pneumonia deployment bundle."""


class LogitExportWrapper(nn.Module):
    """Expose the binary logit with an explicit ``[batch, 1]`` shape for ONNX consumers."""

    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Return raw logits (no sigmoid) shaped ``[batch, 1]``."""
        return self.model(pixel_values).unsqueeze(1)


def sha256_of_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Compute the SHA-256 checksum of a file without loading it entirely in memory."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def export_checkpoint(
    model: nn.Module,
    best_epoch: int,
    best_val_loss: float,
    filename: str = CHECKPOINT_FILENAME,
) -> Path:
    """Write the deployment checkpoint with the best weights and full metadata."""
    path = safe_artifact_path(filename)
    torch.save(
        {
            "schema_version": SCHEMA_VERSION,
            "service_name": SERVICE_NAME,
            "architecture": ARCHITECTURE,
            "model_state_dict": model.state_dict(),
            "best_epoch": int(best_epoch),
            "best_val_loss": float(best_val_loss),
            "class_mapping": {str(index): name for index, name in CLASS_NAMES.items()},
            "positive_class_index": POSITIVE_CLASS_INDEX,
            "threshold": float(DECISION_THRESHOLD),
            "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
            "input_size": [3, IMAGE_SIZE, IMAGE_SIZE],
            "hyperparameters": HYPERPARAMETERS,
        },
        path,
    )
    return path


def export_onnx_model(
    model: nn.Module,
    filename: str = ONNX_FILENAME,
    opset: int = ONNX_OPSET,
) -> Path:
    """Export the model to ONNX with a dynamic batch axis and validate the graph."""
    import onnx

    path = safe_artifact_path(filename)
    export_model = copy.deepcopy(model).to("cpu").eval()
    wrapper = LogitExportWrapper(export_model).eval()
    dummy_input = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)

    with torch.no_grad():
        reference = wrapper(dummy_input)
    assert reference.shape == (1, 1), (
        f"The ONNX wrapper must output [batch, 1], got {tuple(reference.shape)}."
    )

    torch.onnx.export(
        wrapper,
        dummy_input,
        str(path),
        input_names=[ONNX_INPUT_NAME],
        output_names=[ONNX_OUTPUT_NAME],
        dynamic_axes={ONNX_INPUT_NAME: {0: "batch"}, ONNX_OUTPUT_NAME: {0: "batch"}},
        opset_version=opset,
        do_constant_folding=True,
    )

    onnx_model = onnx.load(str(path))
    onnx.checker.check_model(onnx_model)
    return path


def build_deployment_config() -> dict[str, Any]:
    """Build the inference contract consumed by the future ED2A pneumonia service."""
    return {
        "schema_version": SCHEMA_VERSION,
        "service_name": SERVICE_NAME,
        "model_architecture": ARCHITECTURE,
        "framework": "pytorch -> onnx",
        "task": {
            "type": "binary_image_classification",
            "description": (
                "Single-label binary classification of a frontal chest radiograph into "
                "NO_PNEUMONIA or PNEUMONIA."
            ),
        },
        "dataset": {
            "name": DATASET_NAME,
            "annotation_file": "stage_2_train_labels.csv",
            "label_semantics": LABEL_SEMANTICS,
            "aggregation": "one binary record per patientId using max(Target)",
            "external_test_set": (
                "not evaluated: official Stage 2 test images are unlabelled"
            ),
        },
        "class_mapping": {str(index): name for index, name in CLASS_NAMES.items()},
        "positive_class_index": POSITIVE_CLASS_INDEX,
        "supported_upload_formats": ["JPEG", "PNG", "WebP"],
        "input": {
            "input_size": [IMAGE_SIZE, IMAGE_SIZE],
            "channels": 3,
            "channel_order": "RGB",
            "grayscale_to_rgb": (
                "The upload is converted to grayscale and then replicated into three "
                "identical RGB channels, matching the ImageNet-pretrained backbone."
            ),
            "resize": f"bilinear resize to {IMAGE_SIZE}x{IMAGE_SIZE} without cropping",
            "exif_handling": "EXIF orientation is normalised with ImageOps.exif_transpose",
            "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
            "value_range_before_normalization": [0.0, 1.0],
        },
        "onnx": {
            "file": ONNX_FILENAME,
            "opset": ONNX_OPSET,
            "input_name": ONNX_INPUT_NAME,
            "output_name": ONNX_OUTPUT_NAME,
            "input_shape": ["batch", 3, IMAGE_SIZE, IMAGE_SIZE],
            "output_shape": ["batch", 1],
            "dynamic_axes": {"input": "batch", "logit": "batch"},
        },
        "output": {
            "is_logit": True,
            "note": "The model output is a raw logit; no activation is applied inside the graph.",
            "activation_required": "sigmoid",
            "probability_definition": "pneumonia_probability = sigmoid(logit)",
            "probability_threshold": float(DECISION_THRESHOLD),
            "decision_rule": (
                f"class_index = 1 (PNEUMONIA) when sigmoid(logit) >= {DECISION_THRESHOLD}, "
                "otherwise 0 (NO_PNEUMONIA)"
            ),
        },
        "training": {
            "train_fraction": TRAIN_FRACTION,
            "validation_fraction": VALIDATION_FRACTION,
            "split_strategy": "patient-level stratified split on the aggregated binary label",
            "seed": SEED,
            "hyperparameters": HYPERPARAMETERS,
        },
        "disclaimer": (
            "Educational decision-support prototype. Not a real medical diagnosis. "
            + LABEL_SEMANTICS
        ),
    }


def export_config(config: dict[str, Any], filename: str = CONFIG_FILENAME) -> Path:
    """Write the deployment configuration as JSON."""
    path = safe_artifact_path(filename)
    path.write_text(json.dumps(config, indent=2, sort_keys=False), encoding="utf-8")
    return path


def export_metrics(
    metrics: dict[str, Any],
    best_epoch: int,
    best_val_loss: float,
    training_summary: dict[str, Any],
    filename: str = METRICS_FILENAME,
) -> Path:
    """Write the real post-training validation metrics as JSON."""
    path = safe_artifact_path(filename)
    payload = {
        "schema_version": SCHEMA_VERSION,
        "service_name": SERVICE_NAME,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "evaluation_split": "validation (30% patient-level stratified hold-out)",
        "threshold": metrics["threshold"],
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "validation_loss": metrics["validation_loss"],
        "metrics": {
            "accuracy": metrics["accuracy"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "precision": metrics["precision"],
            "recall_sensitivity": metrics["recall_sensitivity"],
            "specificity": metrics["specificity"],
            "f1": metrics["f1"],
            "roc_auc": metrics["roc_auc"],
            "pr_auc_average_precision": metrics["pr_auc_average_precision"],
        },
        "confusion_matrix": metrics["confusion_matrix"],
        "class_support": {
            CLASS_NAMES[0]: metrics["support_negative"],
            CLASS_NAMES[1]: metrics["support_positive"],
            "total": metrics["n_samples"],
        },
        "undefined_metrics": metrics["undefined_metrics"],
        "training_summary": training_summary,
    }
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return path


def export_requirements(filename: str = REQUIREMENTS_FILENAME) -> Path:
    """Write the minimal runtime dependencies of the future inference service.

    Inference runs on ONNX Runtime, so neither torch nor torchvision is required
    in production.
    """
    path = safe_artifact_path(filename)
    lines = [
        "# Minimal runtime dependencies for the ED2A advanced_pneumonia inference service.",
        "# Inference uses ONNX Runtime; torch and torchvision are training-only.",
        "numpy>=1.24",
        "onnxruntime>=1.17",
        "pillow>=10.0",
        "",
    ]
    path.write_text("\n".join(lines), encoding="utf-8")
    return path


def export_manifest(filenames: Sequence[str], filename: str = MANIFEST_FILENAME) -> Path:
    """Write a manifest with generation metadata, file sizes and SHA-256 checksums."""
    path = safe_artifact_path(filename)
    files: dict[str, dict[str, Any]] = {}
    for name in filenames:
        artifact_path = safe_artifact_path(name)
        if not artifact_path.exists():
            raise FileNotFoundError(f"Cannot add missing artifact to the manifest: {artifact_path}")
        files[name] = {
            "bytes": int(artifact_path.stat().st_size),
            "sha256": sha256_of_file(artifact_path),
        }
    payload = {
        "schema_version": SCHEMA_VERSION,
        "service_name": SERVICE_NAME,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "library_versions": library_versions(),
        "artifacts": sorted(files),
        "files": files,
        "reserved_ddxplus_filenames": sorted(DDXPLUS_RESERVED_FILENAMES),
    }
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return path


def create_artifact_bundle(filenames: Sequence[str], filename: str = BUNDLE_FILENAME) -> Path:
    """Package the deployable artifacts into a single ZIP archive."""
    bundle_path = safe_artifact_path(filename)
    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in filenames:
            member_path = safe_artifact_path(name)
            if not member_path.exists():
                raise FileNotFoundError(f"Cannot bundle a missing artifact: {member_path}")
            archive.write(member_path, arcname=name)
    return bundle_path

In [ ]:
"""Run the export (requires a completed training run and the evaluated metrics)."""

best_epoch = int(training_summary["best_epoch"])
best_val_loss = float(training_summary["best_val_loss"])

checkpoint_path = export_checkpoint(model, best_epoch, best_val_loss)
onnx_path = export_onnx_model(model)
deployment_config = build_deployment_config()
config_path = export_config(deployment_config)
metrics_path = export_metrics(validation_metrics, best_epoch, best_val_loss, training_summary)
requirements_path = export_requirements()
manifest_path = export_manifest(BUNDLE_MEMBERS)
bundle_path = create_artifact_bundle(BUNDLE_MEMBERS)

for path in (checkpoint_path, onnx_path, config_path, metrics_path,
             requirements_path, manifest_path, bundle_path):
    print(f"{path.name:>42}  {path.stat().st_size:>12,} bytes")
print()
print(f"Artifact directory: {ARTIFACT_DIR.resolve()}")

---
## 16. Deployment preprocessing and ONNX smoke test

The deployment path is deliberately separate from the training path but reuses the **same**
deterministic evaluation transform, so an uploaded JPEG/PNG/WebP is processed exactly like a
validation image. No training augmentation is ever applied at inference time.

The smoke test loads the exported ONNX graph with ONNX Runtime, preprocesses one validation
image through the deployment function, compares the PyTorch and ONNX logits and probabilities
within a numerical tolerance, and verifies that every artifact file and required JSON field is
present.

In [ ]:
"""Deterministic deployment preprocessing and response formatting."""

# The deployment transform is the validation transform: one definition, no drift.
DEPLOYMENT_TRANSFORMS = EVAL_TRANSFORMS


def preprocess_uploaded_image(image_data: bytes | str | Path) -> torch.Tensor:
    """Convert an uploaded JPEG/PNG/WebP image into a ``[1, 3, 224, 224]`` tensor.

    Args:
        image_data: Raw bytes coming from a multipart upload, or a path to an image file.

    Returns:
        A normalised float tensor ready for the pneumonia model.

    Raises:
        ValueError: If the image cannot be decoded or its format is unsupported.
    """
    try:
        if isinstance(image_data, (str, Path)):
            image = Image.open(Path(image_data))
        else:
            image = Image.open(io.BytesIO(image_data))
        image.load()
    except Exception as exc:  # noqa: BLE001 - uploads must fail with a controlled error
        raise ValueError(f"The uploaded file could not be decoded as an image: {exc}") from exc

    image_format = (image.format or "").upper()
    if image_format not in SUPPORTED_UPLOAD_FORMATS:
        raise ValueError(
            f"Unsupported image format '{image_format or 'unknown'}'. "
            f"Supported formats: {', '.join(SUPPORTED_UPLOAD_FORMATS)}."
        )

    oriented = ImageOps.exif_transpose(image)  # remove EXIF orientation ambiguity
    grayscale = oriented.convert("L")          # radiographs are single-channel
    rgb_image = grayscale.convert("RGB")       # three identical channels for ResNet50

    tensor = DEPLOYMENT_TRANSFORMS(rgb_image).unsqueeze(0)
    assert tensor.shape == (1, 3, IMAGE_SIZE, IMAGE_SIZE), (
        f"Deployment preprocessing must return [1, 3, {IMAGE_SIZE}, {IMAGE_SIZE}], "
        f"got {tuple(tensor.shape)}."
    )
    return tensor


def stable_sigmoid(value: float) -> float:
    """Numerically stable scalar sigmoid."""
    if value >= 0.0:
        return 1.0 / (1.0 + math.exp(-value))
    exponential = math.exp(value)
    return exponential / (1.0 + exponential)


def logit_to_response(
    logit: float,
    threshold: float = DECISION_THRESHOLD,
    model_name: str = SERVICE_NAME,
    class_mapping: dict[int, str] = CLASS_NAMES,
) -> dict[str, Any]:
    """Convert a raw ONNX logit into the JSON-compatible ED2A response structure."""
    probability = stable_sigmoid(float(logit))
    class_index = 1 if probability >= threshold else 0
    return {
        "label": class_mapping[class_index],
        "class_index": class_index,
        "pneumonia_probability": round(probability, 6),
        "threshold": float(threshold),
        "model_name": model_name,
    }

In [ ]:
"""Post-export smoke test: ONNX Runtime parity and artifact verification."""

import onnxruntime as ort

REQUIRED_CONFIG_FIELDS = (
    "schema_version",
    "service_name",
    "model_architecture",
    "task",
    "dataset",
    "class_mapping",
    "positive_class_index",
    "supported_upload_formats",
    "input",
    "onnx",
    "output",
    "training",
)
REQUIRED_METRICS_FIELDS = (
    "schema_version",
    "service_name",
    "threshold",
    "best_epoch",
    "best_val_loss",
    "metrics",
    "confusion_matrix",
    "class_support",
)
REQUIRED_MANIFEST_FIELDS = (
    "schema_version",
    "generated_at_utc",
    "library_versions",
    "artifacts",
    "files",
)


def verify_exported_artifacts() -> dict[str, Any]:
    """Check that every artifact exists and that the JSON contracts are complete."""
    expected = [*BUNDLE_MEMBERS, MANIFEST_FILENAME, BUNDLE_FILENAME]
    missing = [name for name in expected if not (ARTIFACT_DIR / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing exported artifacts: {missing}")

    config_payload = json.loads((ARTIFACT_DIR / CONFIG_FILENAME).read_text(encoding="utf-8"))
    metrics_payload = json.loads((ARTIFACT_DIR / METRICS_FILENAME).read_text(encoding="utf-8"))
    manifest_payload = json.loads((ARTIFACT_DIR / MANIFEST_FILENAME).read_text(encoding="utf-8"))

    for payload, fields, label in (
        (config_payload, REQUIRED_CONFIG_FIELDS, CONFIG_FILENAME),
        (metrics_payload, REQUIRED_METRICS_FIELDS, METRICS_FILENAME),
        (manifest_payload, REQUIRED_MANIFEST_FIELDS, MANIFEST_FILENAME),
    ):
        absent = [name for name in fields if name not in payload]
        if absent:
            raise KeyError(f"{label} is missing required fields: {absent}")

    assert config_payload["service_name"] == SERVICE_NAME
    assert config_payload["output"]["is_logit"] is True
    assert config_payload["output"]["activation_required"] == "sigmoid"

    for name, entry in manifest_payload["files"].items():
        artifact_path = ARTIFACT_DIR / name
        assert entry["bytes"] == artifact_path.stat().st_size, f"Size mismatch for {name}."
        assert entry["sha256"] == sha256_of_file(artifact_path), f"Checksum mismatch for {name}."

    for reserved in DDXPLUS_RESERVED_FILENAMES:
        assert not (ARTIFACT_DIR / reserved).exists(), (
            f"A DDXPlus artifact name was created by mistake: {reserved}"
        )

    return {"config": config_payload, "metrics": metrics_payload, "manifest": manifest_payload}


verified = verify_exported_artifacts()
print("All expected artifacts exist and every required JSON field is present.")

session = ort.InferenceSession(str(ARTIFACT_DIR / ONNX_FILENAME), providers=["CPUExecutionProvider"])
onnx_input = session.get_inputs()[0]
onnx_output = session.get_outputs()[0]
print(f"ONNX input  : {onnx_input.name} {onnx_input.shape}")
print(f"ONNX output : {onnx_output.name} {onnx_output.shape}")
assert onnx_input.name == ONNX_INPUT_NAME and onnx_output.name == ONNX_OUTPUT_NAME
assert len(onnx_output.shape) == 2 and onnx_output.shape[1] == 1, (
    f"The ONNX output must be [batch, 1], got {onnx_output.shape}."
)

# One validation image, encoded to PNG so it travels the real upload path.
sample_row = validation_df.iloc[0]
sample_image = dicom_to_rgb_image(Path(sample_row["dicom_path"]))
png_buffer = io.BytesIO()
sample_image.save(png_buffer, format="PNG")
input_tensor = preprocess_uploaded_image(png_buffer.getvalue())

model.eval()
with torch.inference_mode():
    torch_logit = float(model(input_tensor.to(DEVICE)).reshape(-1)[0].item())
onnx_logit = float(
    session.run([ONNX_OUTPUT_NAME], {ONNX_INPUT_NAME: input_tensor.numpy()})[0].reshape(-1)[0]
)

torch_probability = stable_sigmoid(torch_logit)
onnx_probability = stable_sigmoid(onnx_logit)
print()
print(f"PyTorch logit / probability : {torch_logit:.6f} / {torch_probability:.6f}")
print(f"ONNX    logit / probability : {onnx_logit:.6f} / {onnx_probability:.6f}")
print(f"Absolute logit difference   : {abs(torch_logit - onnx_logit):.3e}")

np.testing.assert_allclose(onnx_logit, torch_logit, rtol=1e-3, atol=1e-3)
np.testing.assert_allclose(onnx_probability, torch_probability, rtol=1e-3, atol=1e-3)

response = logit_to_response(onnx_logit)
print()
print("Example ED2A response payload:")
print(json.dumps(response, indent=2))
print()
print("Smoke test passed: PyTorch and ONNX agree within tolerance.")

---
## 17. ED²A integration contract

The artifacts produced by section 15 are designed for a **future** ED²A `advanced_pneumonia`
service. Copying them into the application is **not** enough to make that service operational.

### What the current API ZIP actually contains

- `API_App/app.py` builds the health payload in `_health_payload()`, where
  `models.advanced_pneumonia` is **hard-coded** to
  `{"ready": False, "reason": "Advanced Pneumonia Model is not implemented or loaded."}`.
- `API_App/model_service.py` is a **DDXPlus-only** service: it loads `best_model.pkl`,
  `preprocessor.pkl` and `label_encoder.pkl` with `joblib` and calls `predict_proba` on a
  tabular feature frame. It has no notion of images, tensors or ONNX.
- `API_App/config.py` exposes only `model_path`, `preprocessor_path`, `label_encoder_path`
  and `metrics_path` under `MODEL_DIR`; there is **no** path for a pneumonia artifact directory.
- `API_App/templates/index.html` renders the "Chest X-ray Assessment" block with a
  `#chest-xray-file` input restricted to JPEG/PNG/WebP (max 10 MB), but the prediction form
  (`<form method="post" action="/web-predict">`) is **not** `multipart/form-data` and never
  submits the selected file. The UI permanently shows
  *"Advanced Pneumonia Model unavailable — the image has not been analysed"*.
- There is **no** image-prediction endpoint anywhere in `app.py`.
- `API_App/requirements.txt` has no `onnxruntime` and no `pillow`.

### What still has to be implemented (outside the scope of this notebook)

1. **A dedicated pneumonia model service** — a new module (for example
   `pneumonia_service.py`) that loads `advanced_pneumonia_model.onnx` with ONNX Runtime and
   reads `advanced_pneumonia_config.json`, kept strictly separate from `ModelService`.
2. **Configuration paths for the new artifact directory** — a new setting (for example
   `PNEUMONIA_MODEL_DIR`) in `config.Settings`, pointing at a directory that is **not** the
   DDXPlus `artifacts/` folder, plus the corresponding `missing_paths` reporting.
3. **An image inference endpoint** — for example `POST /predict-pneumonia`, returning the
   payload produced by `logit_to_response()`: `label`, `class_index`,
   `pneumonia_probability`, `threshold`, `model_name`.
4. **Multipart upload handling** — `UploadFile` validation for JPEG/PNG/WebP, the 10 MB limit
   already advertised by the frontend, and the deterministic preprocessing of section 16.
   `python-multipart` is already a dependency; `onnxruntime` and `pillow` are not.
5. **Health-status integration** — replace the hard-coded `advanced_pneumonia` block in
   `_health_payload()` with the real readiness of the new service, so the existing
   `data-model-status="advanced_pneumonia"` badge in `index.html` can turn "Operational".
6. **Frontend submission of the selected image** — add `enctype="multipart/form-data"` to the
   prediction form (or post the file with `fetch`), then render the returned probability,
   label and disclaimer next to the differential-diagnosis result.

None of these API changes are implemented here: this notebook only produces and validates the
artifacts that such a service would consume.

### Artifact-name safety

The pneumonia bundle never uses `best_model.pkl`, `preprocessor.pkl`, `label_encoder.pkl`,
`model_metrics.json` or `artifact_manifest.json`. Those files belong to the deployed DDXPlus
differential-diagnosis model, `safe_artifact_path()` refuses them, and the smoke test asserts
that none of them was created inside `ARTIFACT_DIR`.